# Prepare model-only RGB resources from the existing dataset

This administrative preparation preserves the original development1000, calibration600, internal-lock400 and official seen/unseen memberships. It reads the existing mounted indexed dataset. It creates a private source index and a physically separate opaque RGB package. No neural inference or scoring occurs.

The official source master is mixed administrative metadata. This trusted adapter projects scene identity, object count and RGB addresses only; it never reads anomaly-label files or scene2D/3D metadata. Keep the private index and source vault out of the model package and public reports.

In [ ]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-native-rgb-release.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    BOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)
    CODE_PROVENANCE = BOOTSTRAP['provenance']
    ROOT = CODE_ROOT
    # Scientific imports are allowed only after the exact runtime/source preflight.
    from IPython.display import display, Image
    import pandas as pd
    display({'source': CODE_PROVENANCE, 'runtime': BOOTSTRAP['runtime'],
             'preparation_only': 'Verified source/runtime does not establish protected-resource readiness'})
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

## Resolve inputs and select one resource operation

Set VPI_NEXT_CONFIG to the stage JSON. `role` is development, calibration, internal_lock, seen or unseen. `action=index` requires raw_root and, for original roles, selection_root and administrative_root. Exact source/admin checksums and memberships are checked. Indexing precedes exposure preparation and performs no RGB extraction.

`action=release` requires rgb_index_root, raw_root, output_root and current method/exposure prerequisites. Original internal-lock additionally needs calibration_receipt_root; official roles need freeze_root, decision_root and official_exposure_root. The release contains only canonical RGB files and an opaque manifest. `maximum_new_scenes` bounds work per call.

In [ ]:
CONFIG=_runtime.resolve_config('native-rgb',ROOT)
ACTION=CONFIG.get('action','index')
from current_stage_execution import run_stage
from vpi_next_evaluation.native_rgb import prepare_rgb_index,release_rgb
if ACTION not in {'index','release'}: raise ValueError('Choose index or release')
OPERATION=prepare_rgb_index if ACTION=='index' else release_rgb
RESULT=run_stage('native-rgb-'+ACTION,CONFIG,lambda:OPERATION(CONFIG,ROOT),CODE_PROVENANCE)
display(RESULT)

## Retain the correct resource

For index, retain the private rgb_index.json plus artifact_lock.json and run_binding.json. Its raw source paths are administrative only. For release, mount output_root/model-input in the next model stage. Keep release receipts and atomic checkpoints in the private parent directory. A final release_lock.json appears only when every original scene is complete.

The release checks free disk space and maximum_output_bytes (default18GiB) for retained checkpoints, released copies and metadata before reading RGB bytes. Completed/resumed outputs must also fit that retained-output budget. Changed input hashes, corrupt media, wrong membership, extra model-package files and failed exposure prerequisites stop. Recovery exports must actually be retained with their sidecars; local checkpoints alone do not guarantee cross-session recovery.